# Day 14 — RAG Part 1: Retrieval
## 30 Days of AI: From NLP to LLMs

---

On Day 13 you built a semantic search engine that finds passages
by meaning. On Days 11-12 you learned to call LLMs and write
effective prompts. Today you build the first half of RAG:
the retrieval pipeline that turns raw documents into a searchable
vector store.

RAG — Retrieval-Augmented Generation — solves the two biggest
problems with LLMs in production:

```
Problem 1: LLMs hallucinate
  → They fill knowledge gaps with confident-sounding wrong answers
  → RAG grounds the model in retrieved facts from YOUR documents

Problem 2: LLMs have a knowledge cutoff
  → GPT-4 does not know about events after its training date
  → RAG gives the model access to current, private, or domain data
```

Today is entirely about the R in RAG. You will go from raw text
files to a queryable vector store — the same pipeline used in
production document Q&A systems.

---

### What You Will Learn Today

- The full RAG architecture — retrieval side explained in depth
- Document loading — text, PDF, and web content
- Chunking strategies — fixed size, sentence, recursive
- Chunk overlap and why it matters
- Metadata — attaching source, page, and section to every chunk
- Building and persisting a vector store
- Retrieval evaluation — precision, recall, hit rate
- Re-ranking — improving retrieval quality after initial search

### Goal by End of Day

Ingest a multi-document corpus, chunk it intelligently, embed
every chunk with metadata, and build a vector store you can
query. Measure retrieval quality. Tomorrow you connect this
to an LLM to complete the full RAG pipeline.

In [ ]:
## Run once
## !pip install sentence-transformers faiss-cpu numpy matplotlib -q

import os
import re
import json
import time
import pickle
import hashlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer

try:
    import faiss
    FAISS_AVAILABLE = True
    print('FAISS    : available')
except ImportError:
    FAISS_AVAILABLE = False
    print('FAISS    : not installed (numpy fallback will be used)')

print('All imports ready.')

---

## Part 1 — The Full RAG Architecture

```
RAG has two phases — offline ingestion and online query.

═══════════════════════════════════════════════════════════════
OFFLINE INGESTION PIPELINE  (run once, or on document updates)
═══════════════════════════════════════════════════════════════

Raw Documents (PDF, TXT, HTML, DOCX, database rows...)
        │
        ▼
   Document Loader
   → Extract plain text from each source
   → Attach source metadata (filename, URL, date, author)
        │
        ▼
   Text Chunker
   → Split documents into overlapping chunks of ~300 tokens
   → Preserve metadata on each chunk
        │
        ▼
   Embedding Model
   → Convert each chunk to a dense vector
        │
        ▼
   Vector Store (FAISS / Chroma / Pinecone / Weaviate)
   → Store vectors + original text + metadata
   → Persist to disk

═══════════════════════════════════════════════════════════════
ONLINE QUERY PIPELINE  (runs per user question, must be fast)
═══════════════════════════════════════════════════════════════

User Question
        │
        ▼
   Query Embedding
   → Embed the question with the SAME model used for documents
        │
        ▼
   Retrieval
   → Find top-k chunks by cosine similarity
   → Optionally re-rank results
        │
        ▼
   Context Assembly
   → Format retrieved chunks into a prompt context block
        │
        ▼
   LLM Generation  ← Day 15
   → LLM reads context + question, generates grounded answer
        │
        ▼
   Answer + Sources
```

Today builds everything above the "Day 15" line.

---

## Part 2 — Document Loading

The first step is extracting plain text from raw documents.
Different source formats require different loaders.

```
Source Type      Library / Method
──────────────────────────────────────────────────────────
.txt files       open() — trivial
.pdf files       PyMuPDF (fitz), pdfplumber, PyPDF2
.docx files      python-docx
.html / web      BeautifulSoup, trafilatura
.csv / Excel     pandas
Database rows    SQLAlchemy + format as text
Notion / Conf.   API + markdown parsing
Slack / Email    API + preprocessing

Key outputs for each document:
  text     : str     — the raw extracted text
  source   : str     — filename, URL, or database ID
  title    : str     — document title if available
  date     : str     — creation or update date
  page     : int     — page number (for PDFs)
```

In [ ]:
# ---------------------------------------------------------------
# Create a realistic multi-document corpus to work with
# In real projects these would be loaded from disk or APIs
# ---------------------------------------------------------------

@dataclass
class Document:
    """
    Represents a loaded document before chunking.
    text     : full extracted text
    metadata : dict of source info (filename, url, date, etc.)
    """
    text     : str
    metadata : Dict = field(default_factory=dict)


# Six realistic documents from three domains
RAW_DOCUMENTS = [
    Document(
        text="""Machine Learning Fundamentals

Machine learning is a branch of artificial intelligence that enables
computers to learn from data without being explicitly programmed.
Instead of writing rules by hand, we train models on examples.

There are three main types of machine learning. Supervised learning
uses labeled data where the correct answer is known. The model learns
a mapping from inputs to outputs. Common algorithms include linear
regression, decision trees, and neural networks.

Unsupervised learning works with unlabeled data. The model discovers
hidden structure without guidance. Clustering groups similar examples
together. Dimensionality reduction compresses data while preserving
important structure. K-means and PCA are classic examples.

Reinforcement learning trains agents to take actions in an environment
to maximize cumulative reward. The agent explores, receives feedback,
and gradually learns a policy. It powers game-playing AI like AlphaGo
and robot control systems.

Overfitting is the most common failure mode in machine learning.
A model overfits when it learns the training data too well, including
noise and outliers. It performs well on training data but fails on
new, unseen examples. Regularization, dropout, and early stopping
are standard techniques to prevent overfitting.

The bias-variance tradeoff describes the tension between two sources
of error. High bias means the model is too simple and underfits.
High variance means the model is too complex and overfits. The goal
is to find the sweet spot that minimizes total error on new data.""",
        metadata={'source': 'ml_fundamentals.txt', 'title': 'Machine Learning Fundamentals',
                  'domain': 'machine_learning', 'date': '2024-01-15'}
    ),

    Document(
        text="""Deep Learning and Neural Networks

Deep learning is a subfield of machine learning that uses neural
networks with many layers — hence the word 'deep'. Each layer
learns increasingly abstract representations of the input data.

A neural network consists of layers of interconnected nodes called
neurons. Each connection has a weight that is learned during training.
The input layer receives raw data. Hidden layers transform it through
non-linear activation functions. The output layer produces predictions.

Backpropagation is the algorithm that trains neural networks. It
computes the gradient of the loss with respect to every weight using
the chain rule. Gradient descent then updates the weights to reduce
the loss. This process repeats for thousands of iterations.

Convolutional Neural Networks (CNNs) are specialized for image data.
Convolutional layers apply learnable filters that detect local patterns
like edges and textures. Pooling layers reduce spatial dimensions.
CNNs power image classification, object detection, and face recognition.

Recurrent Neural Networks (RNNs) process sequential data by maintaining
a hidden state that carries information across time steps. LSTMs add
gating mechanisms to control what is remembered and forgotten.
Both have largely been replaced by Transformers for most NLP tasks.

Transfer learning reuses a model pretrained on a large dataset as
the starting point for a new task. Fine-tuning the pretrained model
on task-specific data is much more efficient than training from scratch.
ImageNet-pretrained CNNs and BERT-pretrained language models are
two of the most impactful examples of transfer learning.""",
        metadata={'source': 'deep_learning.txt', 'title': 'Deep Learning and Neural Networks',
                  'domain': 'machine_learning', 'date': '2024-01-20'}
    ),

    Document(
        text="""The Transformer Architecture

The Transformer was introduced in the 2017 paper Attention Is All You
Need by Vaswani et al. at Google. It replaced recurrent networks for
most sequence modeling tasks by relying entirely on attention mechanisms.

Self-attention is the core mechanism. For each token in a sequence,
self-attention computes a weighted sum of all other tokens' values.
The weights are determined by how relevant each other token is,
measured by the dot product of query and key vectors.

Multi-head attention runs several attention operations in parallel.
Each head learns to attend to different types of relationships.
One head might track syntactic dependencies while another captures
semantic similarity. Outputs are concatenated and projected.

Positional encoding adds position information to token embeddings.
Sinusoidal functions encode absolute position. Without positional
encoding, the model cannot distinguish word order — 'dog bites man'
and 'man bites dog' would produce identical attention patterns.

The Transformer encoder-decoder architecture suits sequence-to-sequence
tasks like translation. The encoder processes the full input. The decoder
generates output tokens autoregressively, attending to encoder output.

BERT uses only the encoder stack with bidirectional attention.
It is pretrained on masked language modeling and excels at
classification, NER, and question answering tasks.
GPT uses only the decoder stack with causal attention for generation.""",
        metadata={'source': 'transformer_architecture.txt', 'title': 'The Transformer Architecture',
                  'domain': 'nlp', 'date': '2024-02-01'}
    ),

    Document(
        text="""Python Best Practices for Data Science

Writing clean, maintainable Python code is essential for data science
projects that need to scale and be understood by others.

Virtual environments should always be used. They isolate project
dependencies and prevent version conflicts. Use venv or conda to create
an isolated Python installation per project. Always pin dependency
versions in a requirements.txt or environment.yml file.

Vectorized operations with NumPy and Pandas are dramatically faster
than Python for-loops. Avoid iterating over DataFrame rows. Use
.apply(), .map(), or vectorized arithmetic directly on arrays.
Profile your code with cProfile or line_profiler to find bottlenecks.

Notebooks are great for exploration but poor for production. Convert
mature analysis into Python modules and packages. Write unit tests
with pytest. Use type hints for better IDE support and documentation.

Data pipelines should be reproducible. Set random seeds. Log all
hyperparameters and results with tools like MLflow or Weights and
Biases. Version your datasets alongside your code.

Memory efficiency matters for large datasets. Use generators instead
of lists for large sequences. Load data in chunks with pandas
chunksize parameter. Consider Dask or Polars for datasets that
do not fit in memory.""",
        metadata={'source': 'python_best_practices.txt', 'title': 'Python Best Practices for Data Science',
                  'domain': 'programming', 'date': '2024-02-10'}
    ),

    Document(
        text="""Large Language Models: A Practical Guide

Large language models are Transformer-based neural networks trained
on massive text corpora using next-token prediction as the objective.
Models like GPT-4, Claude, and Llama have billions of parameters.

The training process has three stages. Pretraining on internet-scale
text teaches the model language structure and world knowledge.
Supervised fine-tuning on human-written examples teaches instruction
following. RLHF aligns the model to human preferences for helpfulness
and safety.

Prompt engineering is the practice of crafting inputs to elicit
desired outputs. Few-shot examples demonstrate the expected format.
Chain-of-thought prompting improves reasoning by asking the model
to explain its steps before answering.

Temperature controls the randomness of generation. Low temperature
makes output deterministic and factual. High temperature increases
diversity and creativity but can reduce coherence.

Context window length limits how much text the model can process
at once. GPT-4 supports 128K tokens. Claude supports 200K tokens.
Long contexts are expensive and models can struggle to attend to
information in the middle of very long contexts.

Fine-tuning adapts a pretrained LLM to a specific domain or task.
LoRA and QLoRA allow fine-tuning large models with limited GPU memory
by training only small low-rank adapter matrices.""",
        metadata={'source': 'llm_guide.txt', 'title': 'Large Language Models: A Practical Guide',
                  'domain': 'nlp', 'date': '2024-02-15'}
    ),

    Document(
        text="""Retrieval-Augmented Generation (RAG)

RAG combines the knowledge retrieval capability of search systems
with the language generation ability of large language models.
It was introduced by Lewis et al. in 2020 and has become one of
the most important patterns in applied LLM engineering.

The motivation for RAG is clear. LLMs have a knowledge cutoff date
and cannot access private or recent information. They also hallucinate
confidently wrong answers when asked about topics outside their
training data. RAG solves both problems by grounding generation in
retrieved documents.

In a RAG pipeline, user queries are embedded and used to retrieve
semantically similar chunks from a vector store. The retrieved chunks
are inserted into the LLM prompt as context. The model generates an
answer that is grounded in the provided evidence.

Chunking strategy is critical. Chunks should be large enough to
contain complete thoughts but small enough to be specific. Overlapping
chunks ensure that information at chunk boundaries is not lost.

Advanced RAG techniques improve retrieval quality. HyDE generates
a hypothetical answer and uses it as the query instead of the
original question. Re-ranking uses a cross-encoder model to re-score
the initial retrieval candidates for higher precision.

Evaluation of RAG systems uses metrics like context recall, faithfulness,
and answer relevance. The RAGAS framework provides automated evaluation
using an LLM-as-judge approach.""",
        metadata={'source': 'rag_overview.txt', 'title': 'Retrieval-Augmented Generation',
                  'domain': 'nlp', 'date': '2024-02-20'}
    ),
]

print(f'Loaded {len(RAW_DOCUMENTS)} documents')
print()
for doc in RAW_DOCUMENTS:
    words = len(doc.text.split())
    print(f'  {doc.metadata["title"][:45]:<45} {words:>4} words')

---

## Part 3 — Chunking Strategies

Chunking is the most impactful decision in building a RAG pipeline.
The same document, chunked differently, can produce dramatically
different retrieval quality.

```
Strategy 1: Fixed-Size Chunking
────────────────────────────────────────────────────────────
Split every N characters or tokens regardless of content.
Simple and predictable.
Problem: splits in the middle of sentences or paragraphs.

Strategy 2: Sentence Chunking
────────────────────────────────────────────────────────────
Split on sentence boundaries. Group N sentences per chunk.
Preserves semantic completeness of individual sentences.
Problem: sentence length varies wildly — some chunks too short.

Strategy 3: Paragraph Chunking
────────────────────────────────────────────────────────────
Split on blank lines or headings. Each paragraph = one chunk.
Natural content boundary — preserves the author's structure.
Problem: paragraphs can be very long or very short.

Strategy 4: Recursive Character Chunking  ← recommended default
────────────────────────────────────────────────────────────
Try to split on: paragraphs → sentences → words → characters
Use the largest boundary that keeps chunks under max_size.
Fall back to smaller boundaries only when needed.
Preserves semantic coherence as much as possible.

Strategy 5: Semantic Chunking (advanced)
────────────────────────────────────────────────────────────
Embed sentences and split when cosine similarity drops sharply.
Detects topic transitions automatically.
Expensive but produces the most coherent chunks.

Overlap:
  Always add 10-20% overlap between consecutive chunks.
  A sentence that falls on a boundary is fully captured
  in at least one chunk either side of the split.
  Typical: chunk_size=512 tokens, overlap=64 tokens.
```

In [ ]:
# ---------------------------------------------------------------
# Chunking implementations — all four strategies
# ---------------------------------------------------------------

@dataclass
class Chunk:
    """
    A single chunk ready for embedding.
    text     : the chunk text
    metadata : inherited from parent document + chunk-level info
    chunk_id : unique identifier (used to look up original text)
    """
    text     : str
    metadata : Dict = field(default_factory=dict)
    chunk_id : str  = ''

    def __post_init__(self):
        if not self.chunk_id:
            self.chunk_id = hashlib.md5(self.text.encode()).hexdigest()[:8]


def fixed_size_chunker(doc: Document, chunk_size=300, overlap=50) -> List[Chunk]:
    """Split text every chunk_size characters with overlap."""
    text   = doc.text
    chunks = []
    start  = 0
    idx    = 0

    while start < len(text):
        end  = min(start + chunk_size, len(text))
        span = text[start:end].strip()
        if span:
            chunks.append(Chunk(
                text     = span,
                metadata = {**doc.metadata, 'chunk_index': idx,
                             'chunk_strategy': 'fixed_size',
                             'char_start': start, 'char_end': end},
            ))
            idx += 1
        start += chunk_size - overlap

    return chunks


def paragraph_chunker(doc: Document, min_length=100, max_length=800) -> List[Chunk]:
    """Split on blank lines (paragraphs). Merge short paragraphs."""
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', doc.text) if p.strip()]
    chunks     = []
    buffer     = ''
    idx        = 0

    for para in paragraphs:
        if len(buffer) + len(para) < max_length:
            buffer = (buffer + '\n\n' + para).strip()
        else:
            if len(buffer) >= min_length:
                chunks.append(Chunk(
                    text     = buffer,
                    metadata = {**doc.metadata, 'chunk_index': idx,
                                 'chunk_strategy': 'paragraph'},
                ))
                idx += 1
            buffer = para

    if buffer and len(buffer) >= min_length:
        chunks.append(Chunk(
            text     = buffer,
            metadata = {**doc.metadata, 'chunk_index': idx,
                         'chunk_strategy': 'paragraph'},
        ))

    return chunks


def recursive_chunker(doc: Document, chunk_size=400, overlap=60) -> List[Chunk]:
    """
    Recursive character text splitter.
    Tries to split on: double newline → single newline → sentence → word
    """
    separators = ['\n\n', '\n', '. ', '! ', '? ', ' ', '']

    def split_text(text, sep_idx=0):
        """Recursively split text using separators in priority order."""
        if len(text) <= chunk_size or sep_idx >= len(separators):
            return [text] if text.strip() else []

        sep = separators[sep_idx]
        parts = text.split(sep) if sep else list(text)

        merged = []
        current = ''

        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    merged.append(current)
                if len(part) > chunk_size:
                    # Part itself too big — recurse with next separator
                    merged.extend(split_text(part, sep_idx + 1))
                    current = ''
                else:
                    current = part.strip()

        if current:
            merged.append(current)

        return [m for m in merged if m.strip()]

    raw_chunks = split_text(doc.text)

    # Add overlap: prepend last N chars of previous chunk
    chunks = []
    for i, text in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            prev    = raw_chunks[i - 1]
            prefix  = prev[-overlap:].strip()
            text    = prefix + ' ' + text
        chunks.append(Chunk(
            text     = text.strip(),
            metadata = {**doc.metadata, 'chunk_index': i,
                         'chunk_strategy': 'recursive'},
        ))

    return chunks


# ---- Compare chunking strategies on one document ----
test_doc = RAW_DOCUMENTS[0]   # ML Fundamentals

fixed_chunks = fixed_size_chunker(test_doc, chunk_size=300, overlap=50)
para_chunks  = paragraph_chunker(test_doc)
rec_chunks   = recursive_chunker(test_doc, chunk_size=400, overlap=60)

print('Chunking Strategy Comparison — ML Fundamentals document')
print('=' * 60)
print(f'{"Strategy":<25} {"Chunks":>8} {"Avg len":>10} {"Min":>6} {"Max":>6}')
print('-' * 60)

for name, chunks in [
    ('Fixed-size (300c, 50 overlap)', fixed_chunks),
    ('Paragraph',                     para_chunks),
    ('Recursive (400c, 60 overlap)',  rec_chunks),
]:
    lengths = [len(c.text) for c in chunks]
    print(f'{name:<35} {len(chunks):>4} {np.mean(lengths):>10.0f} '
          f'{min(lengths):>6} {max(lengths):>6}')

print()
print('First recursive chunk:')
print('-' * 60)
print(rec_chunks[0].text)

In [ ]:
# ---------------------------------------------------------------
# Visualize chunk length distribution
# ---------------------------------------------------------------

all_rec_chunks = []
for doc in RAW_DOCUMENTS:
    all_rec_chunks.extend(recursive_chunker(doc, chunk_size=400, overlap=60))

lengths = [len(c.text) for c in all_rec_chunks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of chunk lengths
ax1.hist(lengths, bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
ax1.axvline(400, color='orange', linestyle=':', label='Target: 400')
ax1.set_xlabel('Chunk length (characters)')
ax1.set_ylabel('Count')
ax1.set_title('Chunk Length Distribution\n(Recursive Chunker, all 6 documents)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Chunks per document
doc_names  = [d.metadata['title'][:20] for d in RAW_DOCUMENTS]
doc_counts = [len(recursive_chunker(d, chunk_size=400, overlap=60)) for d in RAW_DOCUMENTS]
colors_bar = ['steelblue', 'steelblue', 'tomato', 'forestgreen', 'tomato', 'tomato']

ax2.barh(doc_names, doc_counts, color=colors_bar)
ax2.set_xlabel('Number of chunks')
ax2.set_title('Chunks per Document')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f'Total chunks across all documents: {len(all_rec_chunks)}')
print(f'Average chunk length: {np.mean(lengths):.0f} characters')

---

## Part 4 — Building the Vector Store

The vector store is the heart of the retrieval system.
It stores vectors, original text, and metadata in a structure
that supports fast nearest-neighbor queries.

```
For each chunk:
  1. Assign a unique ID
  2. Embed the text → vector of floats
  3. Store in FAISS index (vector → ID mapping)
  4. Store text + metadata in a separate dict (ID → content)

At query time:
  1. Embed the query
  2. FAISS returns top-k IDs
  3. Look up text + metadata for each ID
  4. Return ranked results
```

In [ ]:
# ---------------------------------------------------------------
# Production-ready vector store with persistence
# ---------------------------------------------------------------

class VectorStore:
    """
    A complete vector store for RAG.

    Features:
      - FAISS index (or numpy fallback)
      - Chunk text and metadata storage
      - Metadata filtering
      - Save / load to disk
      - Retrieval with scores
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embed_model = SentenceTransformer(model_name)
        self.dim         = self.embed_model.get_sentence_embedding_dimension()
        self.chunks      = []        # list of Chunk objects
        self.id_to_idx   = {}        # chunk_id → list index

        if FAISS_AVAILABLE:
            self.index = faiss.IndexFlatIP(self.dim)
        else:
            self._vectors = None

    def ingest(self, documents: List[Document],
               chunker=None, batch_size=64,
               chunk_size=400, overlap=60):
        """
        Full ingestion pipeline:
          documents → chunk → embed → index
        """
        if chunker is None:
            chunker = lambda doc: recursive_chunker(doc, chunk_size, overlap)

        # Step 1: chunk all documents
        all_chunks = []
        for doc in documents:
            all_chunks.extend(chunker(doc))

        print(f'  Chunking done    : {len(all_chunks)} chunks from {len(documents)} documents')

        # Step 2: embed in batches
        texts = [c.text for c in all_chunks]
        t0    = time.time()

        vectors = self.embed_model.encode(
            texts,
            normalize_embeddings = True,
            batch_size           = batch_size,
            show_progress_bar    = True,
        ).astype(np.float32)

        print(f'  Embedding done   : {time.time()-t0:.1f}s  ({len(texts)/(time.time()-t0):.0f} chunks/sec)')

        # Step 3: add to index and storage
        start_idx = len(self.chunks)
        for i, chunk in enumerate(all_chunks):
            self.id_to_idx[chunk.chunk_id] = start_idx + i
            self.chunks.append(chunk)

        if FAISS_AVAILABLE:
            self.index.add(vectors)
        else:
            if self._vectors is None:
                self._vectors = vectors
            else:
                self._vectors = np.vstack([self._vectors, vectors])

        print(f'  Index updated    : {len(self.chunks)} total chunks indexed')

    def search(self, query: str, top_k=5,
               filter_metadata: Optional[Dict] = None) -> List[Dict]:
        """
        Search for the top_k most relevant chunks.

        filter_metadata: dict of {key: value} to filter results
          e.g. {'domain': 'nlp'} returns only NLP domain chunks
        """
        q_vec = self.embed_model.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)

        if FAISS_AVAILABLE:
            # Retrieve more candidates if filtering — some may be filtered out
            k_retrieve  = top_k * 5 if filter_metadata else top_k
            scores, ids = self.index.search(q_vec, k_retrieve)
            scores, ids = scores[0], ids[0]
        else:
            k_retrieve = top_k * 5 if filter_metadata else top_k
            all_scores = self._vectors @ q_vec[0]
            ids        = np.argsort(all_scores)[::-1][:k_retrieve]
            scores     = all_scores[ids]

        results = []
        for score, idx in zip(scores, ids):
            if idx < 0 or idx >= len(self.chunks):
                continue
            chunk = self.chunks[idx]

            # Apply metadata filter
            if filter_metadata:
                if not all(chunk.metadata.get(k) == v
                           for k, v in filter_metadata.items()):
                    continue

            results.append({
                'rank'    : len(results) + 1,
                'score'   : float(score),
                'text'    : chunk.text,
                'metadata': chunk.metadata,
                'chunk_id': chunk.chunk_id,
            })

            if len(results) == top_k:
                break

        return results

    def save(self, directory: str):
        """Persist the vector store to disk."""
        path = Path(directory)
        path.mkdir(parents=True, exist_ok=True)

        # Save chunk data
        with open(path / 'chunks.pkl', 'wb') as f:
            pickle.dump({'chunks': self.chunks, 'id_to_idx': self.id_to_idx}, f)

        # Save FAISS index or numpy vectors
        if FAISS_AVAILABLE:
            faiss.write_index(self.index, str(path / 'index.faiss'))
        else:
            np.save(path / 'vectors.npy', self._vectors)

        print(f'Vector store saved to {directory}/')

    @classmethod
    def load(cls, directory: str, model_name='all-MiniLM-L6-v2'):
        """Load a persisted vector store from disk."""
        path  = Path(directory)
        store = cls(model_name=model_name)

        with open(path / 'chunks.pkl', 'rb') as f:
            data = pickle.load(f)
        store.chunks    = data['chunks']
        store.id_to_idx = data['id_to_idx']

        if FAISS_AVAILABLE and (path / 'index.faiss').exists():
            store.index = faiss.read_index(str(path / 'index.faiss'))
        elif (path / 'vectors.npy').exists():
            store._vectors = np.load(path / 'vectors.npy')

        print(f'Vector store loaded: {len(store.chunks)} chunks')
        return store

    def __len__(self):
        return len(self.chunks)

    def __repr__(self):
        return f'VectorStore({len(self.chunks)} chunks, dim={self.dim})'


print('VectorStore class defined.')

In [ ]:
# ---------------------------------------------------------------
# Ingest all documents into the vector store
# ---------------------------------------------------------------

print('Building vector store from 6 documents...')
print('=' * 55)

store = VectorStore(model_name='all-MiniLM-L6-v2')

store.ingest(
    documents  = RAW_DOCUMENTS,
    chunk_size = 400,
    overlap    = 60,
)

# Save to disk
store.save('/tmp/rag_store')

print()
print(repr(store))

In [ ]:
# ---------------------------------------------------------------
# Run search queries and inspect results
# ---------------------------------------------------------------

test_queries = [
    'What is overfitting and how do I prevent it?',
    'How does self-attention work in transformers?',
    'How do I use memory efficiently in Python?',
    'What is RLHF and why is it used for LLMs?',
    'What chunking strategy should I use for RAG?',
]

print('Vector Store Search Results')
print('=' * 70)

for query in test_queries:
    results = store.search(query, top_k=2)
    print(f'\nQuery  : "{query}"')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["metadata"]["source"]} | '
              f'chunk {r["metadata"]["chunk_index"]}')
        print(f'  {r["text"][:120]}...')

In [ ]:
# ---------------------------------------------------------------
# Metadata filtering — search within a specific domain
# ---------------------------------------------------------------

query = 'How are large language models trained?'

print(f'Query: "{query}"')
print()

# Unfiltered — all domains
print('--- Unfiltered (all domains) ---')
for r in store.search(query, top_k=3):
    print(f'  [{r["score"]:.3f}] domain={r["metadata"]["domain"]:15} | {r["text"][:80]}...')

print()

# Filtered — NLP domain only
print('--- Filtered: domain=nlp only ---')
for r in store.search(query, top_k=3, filter_metadata={'domain': 'nlp'}):
    print(f'  [{r["score"]:.3f}] domain={r["metadata"]["domain"]:15} | {r["text"][:80]}...')

print()
print('Metadata filtering is essential in production:')
print('  - Multi-tenant systems (filter by user_id)')
print('  - Date-restricted search (filter by date >= threshold)')
print('  - Department-specific knowledge bases')

---

## Part 5 — Retrieval Evaluation

Before connecting the LLM, you should verify your retrieval is
actually finding the right content. Bad retrieval = bad RAG
answers, no matter how good the LLM is.

```
Key Retrieval Metrics:
─────────────────────────────────────────────────────────────
Hit Rate @ k  :  Fraction of queries where the correct chunk
                 appears in the top-k results.
                 hit_rate@3 = 0.80 means 80% of queries have
                 the answer somewhere in the top 3.

MRR (Mean Reciprocal Rank):
                 Average of 1/rank for the first correct result.
                 MRR=1.0 means always ranked #1.
                 MRR=0.5 means correct result is usually #2.

Precision @ k :  Of the k retrieved chunks, how many are relevant?
                 precision@3 = 0.67 means 2 of 3 chunks are relevant.

Context Recall:  Does the retrieved context contain all information
                 needed to answer the question? (LLM-as-judge)

How to build a test set:
  1. Take 20-50 representative chunks from your corpus
  2. Write 2-3 questions that each chunk answers
  3. Mark which chunk(s) are the 'gold' answer for each question
  4. Run retrieval and measure hit rate
```

In [ ]:
# ---------------------------------------------------------------
# Build a small evaluation set and measure retrieval quality
# ---------------------------------------------------------------

# Ground truth: (question, source_file_that_contains_the_answer)
EVAL_SET = [
    ('What is the difference between supervised and unsupervised learning?',
     'ml_fundamentals.txt'),
    ('How does backpropagation train a neural network?',
     'deep_learning.txt'),
    ('What is the role of positional encoding in transformers?',
     'transformer_architecture.txt'),
    ('How do I avoid for-loops for better Python performance?',
     'python_best_practices.txt'),
    ('What are the three stages of training an LLM?',
     'llm_guide.txt'),
    ('Why does RAG help reduce hallucinations?',
     'rag_overview.txt'),
    ('What is overfitting and how do I prevent it?',
     'ml_fundamentals.txt'),
    ('How do CNNs process image data?',
     'deep_learning.txt'),
    ('How does BERT differ from GPT in architecture?',
     'transformer_architecture.txt'),
    ('What is HyDE and how does it improve RAG retrieval?',
     'rag_overview.txt'),
]


def evaluate_retrieval(store, eval_set, k_values=[1, 3, 5]):
    """
    Compute hit rate and MRR at various k values.
    A 'hit' = the gold source appears in top-k retrieved chunks.
    """
    results = {k: [] for k in k_values}
    reciprocal_ranks = []
    per_query = []

    for question, gold_source in eval_set:
        retrieved = store.search(question, top_k=max(k_values))
        sources   = [r['metadata']['source'] for r in retrieved]

        # Hit rate at each k
        for k in k_values:
            hit = gold_source in sources[:k]
            results[k].append(int(hit))

        # MRR
        rank = next((i+1 for i, s in enumerate(sources) if s == gold_source), None)
        rr   = 1/rank if rank else 0
        reciprocal_ranks.append(rr)

        per_query.append({
            'question'   : question[:50] + '...',
            'gold_source': gold_source,
            'top1_source': sources[0] if sources else 'none',
            'hit@3'      : gold_source in sources[:3],
            'rank'       : rank,
        })

    return {
        'hit_rates': {k: np.mean(v) for k, v in results.items()},
        'mrr'      : np.mean(reciprocal_ranks),
        'per_query': per_query,
    }


eval_results = evaluate_retrieval(store, EVAL_SET)

print('Retrieval Evaluation Results')
print('=' * 55)
for k, rate in eval_results['hit_rates'].items():
    bar = '█' * int(rate * 30)
    print(f'  Hit Rate @ {k}  : {rate:.3f}  {bar}')
print(f'  MRR           : {eval_results["mrr"]:.3f}')
print()
print('Per-query breakdown:')
print(f'{"Question":<52} {"Gold":<25} {"Hit@3"}')
print('-' * 85)
for r in eval_results['per_query']:
    hit_str = '✓' if r['hit@3'] else '✗'
    rank_str = f'(rank {r["rank"]})' if r['rank'] else '(not found)'
    print(f'{r["question"]:<52} {r["gold_source"]:<25} {hit_str} {rank_str}')

---

## Part 6 — Re-Ranking

Initial retrieval with bi-encoder models (like all-MiniLM) is fast
but approximate. Re-ranking uses a cross-encoder to more accurately
score the relevance of each retrieved chunk.

```
Bi-encoder  (retrieval):  embed query and document SEPARATELY
  → Query embedding is computed once
  → Document embeddings are precomputed offline
  → Fast: dot product at query time
  → Cannot see interactions between query and document tokens

Cross-encoder  (re-ranking):  concatenate [query + document] as input
  → Model sees both query and document together
  → Can attend to exact query terms inside the document
  → Much more accurate — directly models relevance
  → Slow: must run full forward pass per (query, document) pair
  → Cannot precompute — only practical for re-scoring top-k results

Two-stage retrieval pipeline (industry standard):
  Stage 1: Bi-encoder retrieves top-50 candidates   (fast)
  Stage 2: Cross-encoder re-ranks top-50 → return top-5  (slow)
  Net result: quality of cross-encoder, speed of bi-encoder
```

In [ ]:
# ---------------------------------------------------------------
# Re-ranking with a cross-encoder
# Cross-encoder models from sentence-transformers/cross-encoders
# ---------------------------------------------------------------

try:
    from sentence_transformers import CrossEncoder

    # Lightweight cross-encoder: ~22M params, fast on CPU
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(query, candidates, cross_encoder, top_k=3):
        """
        Re-rank candidates using a cross-encoder.
        candidates: list of dicts with 'text' key
        Returns candidates sorted by cross-encoder score.
        """
        pairs  = [[query, c['text']] for c in candidates]
        scores = cross_encoder.predict(pairs)

        ranked = sorted(
            zip(candidates, scores),
            key=lambda x: x[1],
            reverse=True
        )

        results = []
        for rank, (candidate, score) in enumerate(ranked[:top_k]):
            results.append({
                **candidate,
                'rank'            : rank + 1,
                'cross_enc_score' : float(score),
                'bi_enc_score'    : candidate['score'],
            })
        return results

    query = 'How do I prevent my model from overfitting on training data?'

    # Stage 1: bi-encoder retrieves top 10
    candidates = store.search(query, top_k=10)

    # Stage 2: cross-encoder re-ranks to top 3
    reranked = rerank(query, candidates, cross_encoder, top_k=3)

    print(f'Query: "{query}"')
    print()
    print('--- Stage 1: Bi-encoder top-3 ---')
    for r in candidates[:3]:
        print(f'  bi_score={r["score"]:.4f} | {r["text"][:90]}...')

    print()
    print('--- Stage 2: Cross-encoder re-ranked top-3 ---')
    for r in reranked:
        print(f'  ce_score={r["cross_enc_score"]:>8.4f}  bi_score={r["bi_enc_score"]:.4f} | '
              f'{r["text"][:75]}...')

except Exception as e:
    print(f'Cross-encoder not available: {e}')
    print()
    print('Re-ranking concept summary:')
    print('  1. Bi-encoder retrieves top-50 fast candidates')
    print('  2. Cross-encoder scores each (query, doc) pair accurately')
    print('  3. Return top-5 by cross-encoder score')
    print('  4. Net: cross-encoder quality at near bi-encoder speed')

In [ ]:
# ---------------------------------------------------------------
# Context assembly — format retrieved chunks into a prompt block
# This is the final output of the retrieval pipeline.
# Tomorrow (Day 15) this gets passed to the LLM.
# ---------------------------------------------------------------

def assemble_context(results: List[Dict], max_chars=2000) -> str:
    """
    Format retrieved chunks into a prompt context block.
    Truncates if total character count exceeds max_chars.

    Output format:
    [Source 1: filename.txt]
    <chunk text>

    [Source 2: filename.txt]
    <chunk text>
    ...
    """
    context_parts = []
    total_chars   = 0

    for r in results:
        source = r['metadata'].get('source', 'unknown')
        block  = f"[Source: {source}]\n{r['text']}"

        if total_chars + len(block) > max_chars:
            # Truncate this block to fit
            remaining = max_chars - total_chars
            if remaining > 100:
                block = block[:remaining] + '...'
                context_parts.append(block)
            break

        context_parts.append(block)
        total_chars += len(block)

    return '\n\n'.join(context_parts)


query   = 'What is overfitting and how do I prevent it?'
results = store.search(query, top_k=3)
context = assemble_context(results)

print('Assembled Context Block')
print('=' * 65)
print(context)
print()
print('-' * 65)
print(f'Total characters in context: {len(context)}')
print()
print('Tomorrow: this context block gets inserted into the LLM prompt')
print('  prompt = system + context + question → LLM → grounded answer')

---

## Day 14 Summary

```
What you built today:

1.  RAG architecture    →  full offline + online pipeline diagram
2.  Document class      →  text + metadata wrapper
3.  Three chunkers      →  fixed-size, paragraph, recursive
4.  Chunk visualization →  length distribution, chunks per document
5.  VectorStore class   →  ingest, search, filter, save, load
6.  Retrieval eval      →  hit rate @ k, MRR, per-query breakdown
7.  Re-ranking          →  bi-encoder + cross-encoder two-stage pipeline
8.  Context assembly    →  format chunks into an LLM-ready context block

The VectorStore and assemble_context() carry forward.
Day 15 connects them to an LLM to build the full RAG pipeline.

What comes next:

  Day 15 — RAG Part 2: Generation.
  You will write the RAG prompt template, connect the vector
  store to an LLM, handle citation of sources in answers,
  and evaluate end-to-end answer quality.
```

### Self-Check Questions

Answer these before Day 15:

1. Why do we add overlap between chunks? What would go wrong without it?
2. Your hit_rate@1 is 0.60 but hit_rate@5 is 0.95. What does this tell you?
   What would you change?
3. When would you use a cross-encoder re-ranker vs just bi-encoder search?
4. A user asks about a topic that is not in your documents at all.
   What should the RAG system return?
5. Why must the query be embedded with the same model used for documents?